<a href="https://colab.research.google.com/github/di-pal-w/compliance_report/blob/main/p5_(2).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install transformers
!pip install sentence-transformers
!pip install faiss-cpu
!pip install pandas
!pip install torch
!pip install accelerate
!pip install langchain
!pip install bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 71.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.7 MB/s eta 0:00:00


In [ ]:
!pip install pymupdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.7/25.7 MB 59.1 MB/s eta 0:00:00


In [ ]:
import fitz  # PyMuPDF

pdf_path = "/content/meta privacy policy.pdf"

doc = fitz.open(pdf_path)

policy = ""
for page in doc:
    policy += page.get_text()

doc.close()

print(policy)

Privacy Policy
Explore the policy
What is the Privacy Policy and what does it cover?
What information do we collect?
How do we use your information?
How is your information shared on Meta Products or with inte‐
grated partners?
How do we share information with third parties?
How do the Meta Companies work together?
How can you manage or delete your information and exercise your
rights?
How long do we keep your information for?
How do we transfer information?
How do we respond to legal requests, comply with applicable law
and prevent harm?
How will you know that the Policy has changed?
How to contact Meta with questions
Why and how we process your information
Other policies
Terms of Service
Cookies policy
What is the Privacy Policy and what does it
cover?
Effective from 16 December 2025
Highlights
7/3/26, 10:21 AM
Meta Privacy Policy – How Meta collects and uses user data
https://mbasic.facebook.com/privacy/policy/printable/version/25862970456621906/
1/91
This Privacy Policy explains ho

In [ ]:
!pip install -U langchain langchain-text-splitters

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [ ]:
def split_policy(text):

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap=50
    )

    return splitter.split_text(text)

In [ ]:
chunks = split_policy(policy)

print(len(chunks))

319


In [ ]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    "BAAI/bge-base-en-v1.5"
)

def create_embeddings(chunks, model):

    return model.encode(
        chunks,
        convert_to_numpy=True
    )

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/777 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
import faiss
import numpy as np

def build_index(embeddings):

    dimension = embeddings.shape[1]

    index = faiss.IndexFlatL2(dimension)

    index.add(np.array(embeddings))

    return index

In [ ]:
import pandas as pd

checklist = pd.read_csv("/content/dpdp_checklist.csv")

print(checklist.head())

   ID                            DPDP_Compliance_Mandate
0   0  The Data Fiduciary must only process personal ...
1   1  Processing of personal data must be based on e...
2   2  Every request for consent must be accompanied ...
3   3  The notice must explicitly state the personal ...
4   4  The notice must explicitly state the specific ...


In [ ]:
# The embedding_model is already defined in cell 2akf6QSN3pC7, so we don't redefine it here.
# If this cell were to be executed independently, it would need its own definition.

def retrieve(requirement, chunks, index, model, k=3):

    query = model.encode(
        [requirement],
        convert_to_numpy=True
    )

    distances, indices = index.search(query, k)

    return [chunks[i] for i in indices[0]]

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_name = "Qwen/Qwen2.5-7B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

llm_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/27.8k [00:00<?, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

In [ ]:
def analyze(requirement, context):

    prompt = f"""
You are a DPDP Act compliance expert.

Requirement:
{requirement}

Relevant Privacy Policy:
{context}

Return ONLY this format:

Status:
Reason:
Evidence:
"""

    messages = [
        {"role":"user","content":prompt}
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(text, return_tensors="pt").to(llm_model.device)

    output = llm_model.generate(
        **inputs,
        max_new_tokens=250
    )

    return tokenizer.decode(
        output[0],
        skip_special_tokens=True
    )

In [ ]:
import pandas as pd

embeddings = create_embeddings(chunks, embedding_model)

index = build_index(embeddings)

checklist = pd.read_csv("/content/dpdp_checklist.csv")

output_file = "/content/dpdp_analysis_results.txt"

with open(output_file, "w", encoding="utf-8") as out:

    for i, (_, row) in enumerate(checklist.iterrows(), start=1):

        requirement = row["DPDP_Compliance_Mandate"]

        context = retrieve(
            requirement,
            chunks,
            index,
            embedding_model
        )

        answer = analyze(
            requirement,
            "\n\n".join(context)
        )

        # Write to text file
        out.write(f"{i}. {requirement}\n")
        out.write(f"{answer.strip()}\n")
        out.write("\n" + "="*80 + "\n\n")

        # Optional: print progress
        print(f"Completed {i}/{len(checklist)}")

print(f"Results saved to {output_file}")

Completed 1/55
Completed 2/55
Completed 3/55
Completed 4/55
Completed 5/55
Completed 6/55
Completed 7/55
Completed 8/55
Completed 9/55
Completed 10/55
Completed 11/55
Completed 12/55
Completed 13/55
Completed 14/55
Completed 15/55
Completed 16/55
Completed 17/55
Completed 18/55
Completed 19/55
Completed 20/55
Completed 21/55
Completed 22/55
Completed 23/55
Completed 24/55
Completed 25/55
Completed 26/55
Completed 27/55
Completed 28/55
Completed 29/55
Completed 30/55
Completed 31/55
Completed 32/55
Completed 33/55
Completed 34/55
Completed 35/55
Completed 36/55
Completed 37/55
Completed 38/55
Completed 39/55
Completed 40/55
Completed 41/55
Completed 42/55
Completed 43/55
Completed 44/55
Completed 45/55
Completed 46/55
Completed 47/55
Completed 48/55
Completed 49/55
Completed 50/55
Completed 51/55
Completed 52/55
Completed 53/55
Completed 54/55
Completed 55/55
Results saved to /content/dpdp_analysis_results.txt
